# KG Pipeline - Notebook Only (Sections 1-12)

This notebook keeps the first five KG stages entirely in memory and now delegates the heavier analysis stages to reusable Python modules under `src/telegram_scraper/analysis/`.
It fetches Telegram channel messages, translates them, embeds them, and exposes notebook-friendly DataFrames and figure objects without writing to Postgres, Redis, or Pinecone.

Use this when you want to stop before node resolution and projection, then inspect the channel's emotional arc, thematic landscape, messaging cadence, vocabulary shifts over time, co-mentioned political actors, rhetorical framing, and reply-threading behavior.


## Prerequisites

- Telegram credentials in `.env`: `TG_API_ID`, `TG_API_HASH`, `TG_PHONE`, `SESSION_PATH`
- OpenAI credentials in `.env`: `OPENAI_API_KEY`, `KG_TRANSLATION_MODEL`, `EMBEDDING_MODEL`
- Project dependencies installed so `telethon`, `openai`, `pandas`, and notebook support are available
- Optional Section 6 analysis dependencies installed: `transformers`, `torch`, `matplotlib`, `seaborn`, `numpy`
- Optional Section 7 topic-modeling dependencies installed: `umap-learn`, `hdbscan`, `plotly`, `scikit-learn`
- Optional Section 8 lexical-shift dependencies installed: `nltk`, `scikit-learn`, `matplotlib`, `wordcloud`
- Optional Section 9 named-entity dependencies installed: `spacy`, `networkx`, `plotly`, `python-louvain`, `matplotlib`, plus a spaCy English model such as `en_core_web_sm`
- Optional Section 10 cadence dependencies reuse `matplotlib`, `seaborn`, and `numpy` from the Section 6 stack
- Optional Section 11 rhetoric dependencies reuse `transformers`, `torch`, `plotly`, `matplotlib`, and `seaborn` from Sections 6-10
- Optional Section 12 reply-threading dependencies reuse `networkx` from Section 9 plus `scipy` and `matplotlib` from earlier analysis sections

## Write targets

| # | Stage | Writes to |
|---|---|---|
| 1 | Setup | - |
| 2 | Telegram Client & Channel Discovery | local session file, optional media under `tmp/notebook-media/` |
| 3 | Message Ingestion | in-memory Python objects only |
| 4 | Translation | OpenAI API response only, held in memory |
| 5 | Embedding | OpenAI API response only, held in memory |
| 6 | Sentiment & Emotion Over Time | Hugging Face model cache plus in-memory analysis DataFrames and plots |
| 7 | Topic Modeling Landscape | in-memory topic-modeling DataFrames plus interactive Plotly figures |
| 8 | Word Frequency & TF-IDF Shifts | in-memory lexical-analysis DataFrames plus matplotlib figures and word clouds |
| 9 | Named Entity Network Graphs | in-memory entity tables, NetworkX graph object, Plotly figures, and matplotlib ego-network panels |
| 10 | Messaging Cadence & Volume Heatmaps | in-memory cadence DataFrames plus matplotlib / seaborn figures |
| 11 | Framing & Rhetoric Analysis | in-memory rhetoric DataFrames plus Plotly / matplotlib figures |
| 12 | Reply Threading & Engagement Proxy Analysis | in-memory reply-graph DataFrames, NetworkX graph object, and matplotlib figures |

The notebook leaves you with `raw_messages`, `translated_messages`, `messages_df`, `translation_df`, `embedding_lookup`, `embeddings_df`, `df_text`, `sentiment_emotion_df`, `sentiment_window_df`, `emotion_window_df`, `candidate_events_df`, `topic_messages_df`, `topic_keyword_df`, `topic_summary_df`, `topic_prevalence_df`, `topic_daily_share_df`, `topic_example_messages_df`, `topic_scatter_fig`, `topic_prevalence_fig`, `topic_time_fig`, `tfidf_messages_df`, `tfidf_period_docs_df`, `tfidf_score_df`, `tfidf_rank_df`, `tfidf_risers_df`, `tfidf_fallers_df`, `tfidf_movers_df`, `tfidf_top_terms_df`, `tfidf_bar_plot_df`, `tfidf_wordcloud_frequencies`, `tfidf_bump_fig`, `tfidf_terms_fig`, `tfidf_wordcloud_fig`, `entity_messages_df`, `entity_mentions_df`, `entity_summary_df`, `entity_pair_df`, `entity_network_summary_df`, `entity_network_nodes_df`, `entity_network_edges_df`, `entity_community_summary_df`, `named_entity_graph`, `entity_top_entities_fig`, `entity_network_fig`, `entity_ego_fig`, `cadence_messages_df`, `cadence_hourly_counts_df`, `cadence_daily_summary_df`, `cadence_top_spikes_df`, `cadence_calendar_heatmap_fig`, `cadence_structural_rhythm_fig`, `cadence_volume_media_fig`, `rhetoric_df_text`, `rhetoric_messages_df`, `rhetoric_window_df`, `rhetoric_window_long_df`, `rhetoric_label_counts_df`, `rhetoric_half_summary_df`, `rhetoric_half_flow_df`, `rhetoric_taxonomy_df`, `rhetoric_example_messages_df`, `rhetoric_validation_sample_df`, `rhetoric_sentiment_crosstab_df`, `rhetoric_sentiment_share_df`, `rhetoric_frame_label_lookup`, `rhetoric_frame_color_map`, `rhetoric_over_time_fig`, `rhetoric_transition_fig`, `rhetoric_examples_fig`, `rhetoric_sentiment_heatmap_fig`, `reply_messages_df`, `reply_edges_df`, `reply_top_replied_df`, `reply_thread_summary_df`, `reply_distribution_df`, `reply_hourly_reply_rate_df`, `reply_feature_tests_df`, `reply_media_contingency_df`, `reply_first_reply_timing_df`, `reply_content_review_df`, `reply_summary_df`, `reply_graph`, `reply_distribution_fig`, `reply_threads_fig`, `reply_scatter_fig`, and `reply_timing_fig`.


---
## Section 1 - Environment Setup

In [1]:
from __future__ import annotations

import sys
import time
from pathlib import Path

try:
    import nest_asyncio
except ImportError:
    nest_asyncio = None

if nest_asyncio is not None:
    nest_asyncio.apply()

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 180)

_nb_dir = Path.cwd().resolve()
_project_root = _nb_dir if (_nb_dir / "src").exists() else _nb_dir.parent
_src_path = _project_root / "src"
if not _src_path.exists():
    raise RuntimeError(f"Could not locate src/ from notebook cwd: {_nb_dir}")
if str(_src_path) not in sys.path:
    sys.path.insert(0, str(_src_path))

print(f"Project root: {_project_root}")
print(f"Python: {sys.version.split()[0]}")

Project root: /Users/lfpmb/Documents/telegram-twitter-scraper
Python: 3.9.6


In [ ]:
from telegram_scraper.config import Settings
from telegram_scraper.notebook_pipeline import NotebookSettings, OpenAIEmbedder, OpenAIMessageTranslator

_t1 = time.monotonic()

_env_candidates = [_project_root / ".env", _nb_dir / ".env"]
_env_path = next((p for p in _env_candidates if p.exists()), _project_root / ".env")

settings = Settings.load(_env_path)
notebook_settings = NotebookSettings.load(_env_path)

settings.require_credentials()
notebook_settings.require_translation()
notebook_settings.require_embeddings()

_session_path = settings.session_path if settings.session_path.is_absolute() else (_project_root / settings.session_path).resolve()
notebook_media_root = (_project_root / "tmp" / "notebook-media").resolve()
notebook_media_root.mkdir(parents=True, exist_ok=True)

translator = OpenAIMessageTranslator(
    api_key=notebook_settings.openai_api_key,
    model=notebook_settings.translation_model,
    max_chars=notebook_settings.semantic_max_chars,
    batch_size=notebook_settings.semantic_batch_size,
)
embedder = OpenAIEmbedder(
    api_key=notebook_settings.openai_api_key,
    model=notebook_settings.embedding_model,
)

print(f"Loaded settings from: {_env_path}")
print(f"Telegram session path: {_session_path}")
print(f"Notebook media root: {notebook_media_root}")
print(f"Translation model: {notebook_settings.translation_model}")
print(f"Embedding model: {notebook_settings.embedding_model}")
print(f"Section 1 done in {time.monotonic() - _t1:.2f}s")

---
## Section 2 - Telegram Client & Channel Discovery

In [ ]:
from telegram_scraper.chat_discovery import discover_chats, filter_chats
from telegram_scraper.models import ChatType
from telegram_scraper.telegram_client import TelegramAccountClient

_t2 = time.monotonic()

telegram_client = TelegramAccountClient(
    api_id=settings.api_id or 0,
    api_hash=settings.api_hash,
    phone=settings.phone,
    session_path=_session_path,
    output_root=notebook_media_root,
)

await telegram_client.connect()
dialogs = await telegram_client.get_dialogs()
all_chats = discover_chats(dialogs)
channels = filter_chats(
    all_chats,
    chat_types=(ChatType.CHANNEL,),
    include_chats=settings.include_chats,
    exclude_chats=settings.exclude_chats,
)

channels_df = pd.DataFrame(
    [
        {
            "#": i,
            "chat_id": chat.chat_id,
            "title": chat.title or "",
            "username": chat.username or "",
            "slug": chat.slug or "",
        }
        for i, chat in enumerate(channels)
    ]
)

print(f"Found {len(channels)} channels in {time.monotonic() - _t2:.2f}s")
display(channels_df)

In [ ]:
CHANNEL_INDEX = 5
MESSAGE_LIMIT = 1200

if not channels:
    raise RuntimeError("No channels matched the current include/exclude filters.")
if CHANNEL_INDEX < 0 or CHANNEL_INDEX >= len(channels):
    raise IndexError(f"CHANNEL_INDEX={CHANNEL_INDEX} is out of range for {len(channels)} channels.")

selected_chat = channels[CHANNEL_INDEX]
print(f"Selected channel: [{selected_chat.chat_id}] {selected_chat.title or '(untitled)'}")
print(f"Username: @{selected_chat.username}" if selected_chat.username else "Username: ---")
print(f"Message limit: {MESSAGE_LIMIT}")

---
## Section 3 - Message Ingestion

In [ ]:
from telegram_scraper.notebook_pipeline import normalize_message_record

_t3 = time.monotonic()

raw_envelopes = []
async for envelope in telegram_client.iter_message_envelopes(
    selected_chat,
    limit=MESSAGE_LIMIT,
    reverse=False,
):
    raw_envelopes.append(envelope)

raw_messages = [normalize_message_record(env.record, raw_json=env.raw_json) for env in raw_envelopes]
raw_messages.sort(key=lambda message: (message.timestamp, message.message_id))

messages_df = pd.DataFrame(
    [
        {
            "channel_id": message.channel_id,
            "message_id": message.message_id,
            "timestamp": message.timestamp,
            "sender_name": message.sender_name or "",
            "reply_to_message_id": message.reply_to_message_id,
            "text": (message.text or "")[:200],
            "has_media": bool(message.media_refs),
        }
        for message in raw_messages
    ]
)

print(f"Fetched {len(raw_messages)} messages in {time.monotonic() - _t3:.2f}s")
display(messages_df.head(20))

---
## Section 4 - Translation

In [ ]:
_t4 = time.monotonic()

translated_messages = translator.translate_messages(raw_messages)

translation_df = pd.DataFrame(
    [
        {
            "message_id": message.message_id,
            "timestamp": message.timestamp,
            "source_language": message.source_language or "und",
            "original_text": (message.text or "")[:160],
            "english_text": (message.english_text or "")[:160],
            "used_translation": bool(
                (message.english_text or "").strip()
                and (message.english_text or "").strip() != (message.text or "").strip()
            ),
        }
        for message in translated_messages
    ]
)

language_counts_df = (
    translation_df["source_language"]
    .value_counts()
    .rename_axis("source_language")
    .reset_index(name="message_count")
)

print(f"Translated {len(translated_messages)} messages in {time.monotonic() - _t4:.2f}s")
display(language_counts_df)
display(translation_df.head(20))

---
## Section 5 - Embedding

In [ ]:
from telegram_scraper.notebook_pipeline import preferred_message_text, safe_message_text

_t5 = time.monotonic()

embeddable_messages = [
    message
    for message in translated_messages
    if (message.english_text or message.text or "").strip()
]

embed_texts = [
    safe_message_text(
        preferred_message_text(message) or "(media only telegram message)",
        max_chars=notebook_settings.semantic_max_chars,
    )
    for message in embeddable_messages
]

vectors = embedder.embed_texts(embed_texts) if embed_texts else []
embedding_lookup = {
    (message.channel_id, message.message_id): vector
    for message, vector in zip(embeddable_messages, vectors)
}
embedding_records = [
    {
        "channel_id": message.channel_id,
        "message_id": message.message_id,
        "timestamp": message.timestamp,
        "embedding": vector,
    }
    for message, vector in zip(embeddable_messages, vectors)
]

embeddings_df = pd.DataFrame(
    [
        {
            "channel_id": message.channel_id,
            "message_id": message.message_id,
            "timestamp": message.timestamp,
            "embedding_dim": len(vector),
            "embedding_preview": [round(value, 4) for value in vector[:8]],
        }
        for message, vector in zip(embeddable_messages, vectors)
    ]
)

print(f"Embedded {len(embeddable_messages)} messages in {time.monotonic() - _t5:.2f}s")
print(f"Embedding dimensionality: {len(vectors[0]) if vectors else 0}")
display(embeddings_df.head(20))

---
## Section 6 - Sentiment & Emotion Over Time

This section implements `docs/01_sentiment_emotion_over_time.md` directly inside the notebook.
It scores translated English text when available, aggregates sentiment/emotion over time, and produces both the dual-axis timeline and hourly dominant-emotion heatmap.

The first run downloads both Hugging Face models into the local cache, so expect this section to take a few minutes on CPU.

In [ ]:
from telegram_scraper.analysis.sentiment import (
    SentimentEmotionConfig,
    run_sentiment_emotion_analysis,
)

EVENT_ANNOTATIONS = [
    # Replace these placeholders after reviewing candidate_events_df.
    # {"timestamp": "2026-04-10 12:00:00+00:00", "label": "Iran-US peace talks begin"},
    # {"timestamp": "2026-04-11 18:00:00+00:00", "label": "Trump threatens Iranian infrastructure"},
]

sentiment_config = SentimentEmotionConfig()
print(
    f"Section 6 configured with {sentiment_config.window_freq} aggregation windows "
    f"and {sentiment_config.heatmap_freq} heatmap bins."
)


In [ ]:
sentiment_result = run_sentiment_emotion_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    event_annotations=EVENT_ANNOTATIONS,
    config=sentiment_config,
)
globals().update(sentiment_result.to_namespace())

print(f"Section 6 completed in {sentiment_result.analysis_seconds:.2f}s")
if not EVENT_ANNOTATIONS:
    print(
        "EVENT_ANNOTATIONS is empty; using candidate placeholders. "
        "Replace them with reviewed event labels for the final chart."
    )


In [ ]:
display(overall_summary_df)
display(sentiment_label_counts_df)
display(emotion_label_counts_df)
display(candidate_events_df)

In [ ]:
display(
    sentiment_emotion_df[
        [
            "timestamp",
            "sentiment_score",
            "dominant_sentiment",
            "dominant_emotion",
            "emotion_confidence",
            "text",
        ]
    ].head(10)
)

In [ ]:
display(event_annotations_df)

In [ ]:
display(sentiment_over_time_fig)

This chart combines two time scales, so the interpretation cell should stay descriptive rather than hard-code one run's numbers.

**The thin gray sentiment line and the thick black rolling line use 6-hour windows.** Those are the series that show the broader emotional arc over time.

**The outlined callout marker is different: it shows the single most extreme 1-hour mean sentiment.** It is plotted directly at the hourly value it reports, so it may sit above or below the 6-hour curve.

**The stacked area still reflects 6-hour dominant-emotion proportions.** Use it to see whether anger, fear, joy, sadness, or neutral emotion dominates each broader time window.

**When you want a fresh narrative for the current dataset, rerun the section and inspect** `most_extreme_hour`, `candidate_events_df`, `event_annotations_df`, and `sentiment_window_df`.

This avoids stale claims when the notebook is rerun on a different time window or after the event labels are edited.

In [ ]:
display(emotion_heatmap_fig)
display(hourly_dominant_emotion_df.head(24))

---
## Section 7 - Topic Modeling Landscape

This section implements `docs/02_topic_modeling_landscape.md` directly inside the notebook.
It reuses the OpenAI embeddings from Section 5, cleans message text for topic labeling, projects messages into 2D with UMAP, clusters them with HDBSCAN, extracts c-TF-IDF-style keywords, and renders interactive Plotly views of the thematic landscape.

Run Sections 1-5 first. Section 6 is not required for this section.
Topic names default to keyword-based placeholders; edit `TOPIC_LABEL_OVERRIDES` after reviewing the summaries if you want manual labels such as `Diplomacy & Nuclear Negotiations`.


In [ ]:
from telegram_scraper.analysis.topics import TopicModelingConfig, run_topic_modeling_analysis

topic_config = TopicModelingConfig()
print(
    "Section 7 configured with "
    f"UMAP neighbors={topic_config.umap_neighbors}, "
    f"min_dist={topic_config.umap_min_dist}, "
    f"HDBSCAN min_cluster_size={topic_config.min_cluster_size}, "
    f"min_samples={topic_config.min_samples}."
)

In [ ]:
topic_result = run_topic_modeling_analysis(
    translated_messages,
    embedding_lookup,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    config=topic_config,
)
globals().update(topic_result.to_namespace())

print(f"Section 7 completed in {topic_result.analysis_seconds:.2f}s")
if not topic_config.label_overrides:
    print("TOPIC_LABEL_OVERRIDES is empty; labels below are keyword-based placeholders.")

display(topic_prep_summary_df)
display(topic_summary_df)
display(topic_keyword_df.loc[topic_keyword_df["rank"] <= 5])
display(topic_example_messages_df)

In [ ]:
display(topic_scatter_fig)
display(topic_prevalence_fig)
display(topic_time_fig)
display(topic_prevalence_df)

---
## Section 8 - Word Frequency & TF-IDF Shifts

This section implements `docs/03_word_frequency_tfidf_shifts.md` directly inside the notebook.
It builds four time-sliced pseudo-documents from translated text, scores each period with TF-IDF, highlights the largest rank movers between the first and last periods, and renders the bump-chart / faceted-bar views requested in the plan.

Run Sections 1-4 first. Sections 5-7 are not required for this section.


In [ ]:
from telegram_scraper.analysis.lexical import LexicalShiftConfig, run_tfidf_shift_analysis

tfidf_config = LexicalShiftConfig()
print(
    f"Section 8 configured with {tfidf_config.periods} equal time bins and up to "
    f"{tfidf_config.max_features} TF-IDF features."
)

In [ ]:
tfidf_result = run_tfidf_shift_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    config=tfidf_config,
)
globals().update(tfidf_result.to_namespace())

print(f"Section 8 completed in {tfidf_result.analysis_seconds:.2f}s")
display(tfidf_summary_df)
display(tfidf_period_docs_df[["period_label", "message_count", "token_count", "first_seen", "last_seen"]])
display(tfidf_movers_df)
display(tfidf_messages_df[["timestamp", "source_language", "used_translation", "token_count", "clean_text"]].head(10))

In [ ]:
display(tfidf_bump_fig)
display(tfidf_terms_fig)

In [ ]:
display(tfidf_wordcloud_fig)

---
These TF-IDF word clouds are **supplementary** rather than primary analysis.
Use them for a fast visual impression of each period's distinctive vocabulary, then rely on `tfidf_bump_fig`, `tfidf_terms_fig`, and the mover tables for precise comparisons.

Because the clouds are sized by **TF-IDF score instead of raw frequency**, they downweight channel-wide boilerplate and better emphasize terms that are unusually characteristic of a given time slice.


---
## Section 9 - Named Entity Network Graphs


In [ ]:
from telegram_scraper.analysis.entities import NamedEntityConfig, run_named_entity_analysis

entity_config = NamedEntityConfig()
print(
    f"Section 9 configured with spaCy model={entity_config.spacy_model}, "
    f"min_entity_messages={entity_config.min_message_count}, "
    f"and min_edge_weight={entity_config.min_edge_weight}."
)

In [ ]:
entity_result = run_named_entity_analysis(
    df_text,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    config=entity_config,
)
globals().update(entity_result.to_namespace())

print(f"Section 9 completed in {entity_result.analysis_seconds:.2f}s")
display(entity_extraction_summary_df)
display(entity_network_summary_df)
display(entity_summary_df.head(20))
display(entity_pair_df.loc[entity_pair_df["weight"] >= entity_config.min_edge_weight].head(20))
display(entity_community_summary_df)
display(
    entity_messages_df.sort_values(["entity_count", "timestamp"], ascending=[False, True])[
        ["timestamp", "entity_count", "entity_names_preview", "text_preview"]
    ].head(10)
)

In [ ]:
display(entity_top_entities_fig)
display(entity_network_fig)
if entity_ego_fig is not None:
    display(entity_ego_fig)
else:
    print("No ego networks available because the filtered graph is empty.")

---
## Section 10 - Messaging Cadence & Volume Heatmaps

This section implements `docs/05_messaging_cadence_heatmaps.md` directly inside the notebook.
It uses only message timestamps and the `has_media` flag, so it can run after Section 3; if Section 4 has already run, translated text previews are reused for spike labels.
The outputs include the continuous calendar-strip heatmap, the weekday/hour structural rhythm heatmap, the hourly volume + media overlay, and tables for daily summaries plus top spike hours.


In [ ]:
from telegram_scraper.analysis.cadence import MessagingCadenceConfig, run_messaging_cadence_analysis

CADENCE_EVENT_ANNOTATIONS = [
    # Replace these placeholders after reviewing cadence_top_spikes_df / cadence_spike_messages_df.
    # {"timestamp": "2026-04-10 12:00:00+00:00", "label": "Breaking-news burst around nuclear talks"},
    # {"timestamp": "2026-04-11 18:00:00+00:00", "label": "Military escalation coverage spikes"},
]

cadence_config = MessagingCadenceConfig()
print(
    f"Section 10 configured with {cadence_config.hour_freq} bins, "
    f"top {cadence_config.top_spike_candidates} spike candidates, "
    f"and {cadence_config.annotated_spikes} annotated spike cells."
)

In [ ]:
cadence_source_messages = translated_messages if "translated_messages" in globals() else raw_messages

cadence_result = run_messaging_cadence_analysis(
    cadence_source_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    event_annotations=CADENCE_EVENT_ANNOTATIONS,
    config=cadence_config,
)
globals().update(cadence_result.to_namespace())

print(f"Section 10 completed in {cadence_result.analysis_seconds:.2f}s")
if not CADENCE_EVENT_ANNOTATIONS:
    print(
        "CADENCE_EVENT_ANNOTATIONS is empty; using spike-preview placeholders. "
        "Replace them with reviewed labels for the final calendar heatmap."
    )

display(cadence_summary_df)
display(cadence_daily_summary_df)
display(cadence_top_spikes_df)
display(cadence_weekday_observation_df)
display(cadence_spike_messages_df.head(20))

In [ ]:
display(cadence_calendar_heatmap_fig)
display(cadence_structural_rhythm_fig)
display(cadence_volume_media_fig)
display(cadence_media_hourly_df.head(24))

---
## Section 11 - Framing & Rhetoric Analysis

This section implements `docs/06_framing_rhetoric_analysis.md` directly inside the notebook.
It classifies each text-bearing message into a rhetorical frame with a zero-shot MNLI model, aggregates the frame mix over 12-hour windows, compares the first vs. second half of the collection, and surfaces high-confidence examples plus a validation sample for manual review.

Run Sections 1-4 first. Section 6 is optional; if `sentiment_emotion_df` is available, this section also builds the rhetoric × sentiment heatmap.
The first run downloads the zero-shot model into the local cache, and CPU inference can take a while on larger message sets.


In [ ]:
from telegram_scraper.analysis.framing import RhetoricFramingConfig, run_rhetoric_framing_analysis

rhetoric_config = RhetoricFramingConfig()
print(
    f"Section 11 configured with {rhetoric_config.window_freq} windows, "
    f"{rhetoric_config.model_batch_size}-message classifier batches, "
    f"and ambiguous threshold {rhetoric_config.ambiguous_threshold:.2f}."
)

In [ ]:
rhetoric_sentiment_source_df = sentiment_emotion_df if "sentiment_emotion_df" in globals() else None

rhetoric_result = run_rhetoric_framing_analysis(
    translated_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    sentiment_emotion_df=rhetoric_sentiment_source_df,
    config=rhetoric_config,
)
globals().update(rhetoric_result.to_namespace())

print(f"Section 11 completed in {rhetoric_result.analysis_seconds:.2f}s")
if rhetoric_sentiment_source_df is None:
    print("sentiment_emotion_df is unavailable; skipping the rhetoric × sentiment heatmap.")

In [ ]:
display(rhetoric_taxonomy_df)
display(rhetoric_summary_df)
display(rhetoric_label_counts_df)
display(rhetoric_half_summary_df)
display(rhetoric_example_messages_df)
display(rhetoric_validation_sample_df.head(20))
display(
    rhetoric_messages_df[
        [
            "timestamp",
            "dominant_frame",
            "confidence",
            "second_frame",
            "second_confidence",
            "text",
        ]
    ].head(10)
)

In [ ]:
display(rhetoric_over_time_fig)
display(rhetoric_transition_fig)
display(rhetoric_examples_fig)
if rhetoric_sentiment_heatmap_fig is not None:
    display(rhetoric_sentiment_heatmap_fig)
    display(rhetoric_sentiment_crosstab_df)
else:
    print("Rhetoric × sentiment heatmap unavailable because sentiment_emotion_df was not provided.")

The Sankey diagram above is a **distribution reallocation view**, not a message-level trajectory model.
It preserves each half's frame totals and helps you see how the rhetorical mix changes between the first and second halves of the observation window.

For manual QA, inspect `rhetoric_validation_sample_df` and `rhetoric_example_messages_df`.
If a category is systematically misassigned, adjust the candidate label wording in `src/telegram_scraper/analysis/framing.py` and rerun the section.


---
## Section 12 - Reply Threading & Engagement Proxy Analysis

This section implements `docs/07_reply_threading_analysis.md` directly inside the notebook.
It builds a reply graph from `reply_to_message_id`, measures which messages attract follow-up replies, summarizes the largest threaded components, tests simple feature correlates, and surfaces parent→reply pairs for manual qualitative review.

Run Section 3 first. Section 4 is optional for translated previews; Sections 5-6 are optional if you want embedding-based reply similarity and sentiment-colored scatter plots.


In [ ]:
from telegram_scraper.analysis.reply_threading import ReplyThreadingConfig, run_reply_threading_analysis

reply_config = ReplyThreadingConfig()
print(
    f"Section 12 configured with top {reply_config.top_replied_messages} replied-to messages, "
    f"top {reply_config.top_threads} thread diagrams, "
    f"and {reply_config.first_reply_hist_bins} timing bins."
)


In [ ]:
reply_source_messages = translated_messages if "translated_messages" in globals() else raw_messages
reply_sentiment_source_df = sentiment_emotion_df if "sentiment_emotion_df" in globals() else None
reply_embedding_lookup = embedding_lookup if "embedding_lookup" in globals() else None

reply_result = run_reply_threading_analysis(
    reply_source_messages,
    channel_label=selected_chat.title or selected_chat.username or str(selected_chat.chat_id),
    sentiment_emotion_df=reply_sentiment_source_df,
    embedding_lookup=reply_embedding_lookup,
    config=reply_config,
)
globals().update(reply_result.to_namespace())

print(f"Section 12 completed in {reply_result.analysis_seconds:.2f}s")
if reply_sentiment_source_df is None:
    print("sentiment_emotion_df is unavailable; coloring the scatter plot by has_media instead of sentiment.")
if reply_embedding_lookup is None:
    print("embedding_lookup is unavailable; reply-content review falls back to lexical overlap instead of embedding cosine similarity.")


In [ ]:
display(reply_summary_df)
display(reply_feature_tests_df)
display(reply_distribution_df)
display(
    reply_thread_summary_df[
        [
            "thread_label",
            "thread_size",
            "thread_depth",
            "component_total_nodes",
            "external_node_count",
            "reply_messages",
            "messages_receiving_replies",
            "root_message_id",
            "root_in_dataset",
            "root_preview",
            "top_replied_message_id",
            "top_replied_count",
        ]
    ].head(10)
)
display(
    reply_top_replied_df[
        [
            "thread_label",
            "message_id",
            "timestamp",
            "reply_count",
            "message_depth",
            "thread_size",
            "thread_depth",
            "has_media",
            "text_preview",
        ]
    ]
)
display(reply_hourly_reply_rate_df)
display(reply_media_contingency_df)
display(reply_first_reply_timing_df.head(20))
display(reply_content_review_df.head(30))


In [ ]:
display(reply_distribution_fig)
display(reply_threads_fig)
display(reply_scatter_fig)
display(reply_timing_fig)


`reply_content_review_df["relationship_hint"]` is heuristic only.
Use it to prioritize manual reading of parent/reply pairs, not as a final label set.

If `messages_replying_to_missing_parent` is non-zero in `reply_summary_df`, some threads start outside the collection window, so treat the reported thread depth as **window-limited** rather than a full channel-history depth.


---
## Optional Cleanup

Run the next cell when you are done with the Telegram client.

In [ ]:
await telegram_client.disconnect()
print("Telegram client disconnected.")